# ESCan ffDTF Batch (hyperscanning)

This notebook computes batch ffDTF in a hyperscanning setup:
- same EEG channel subset for child and caregiver,
- one joint MVAR system over all selected channels from both persons,
- ffDTF integration in a selected frequency band,
- real dyads and surrogate dyads (caregiver from one dyad + random child from another dyad),
- child-derived group label for both real and surrogate rows,
- one output dataframe with columns: dyadID, surrogate_pair_id, surrogate_caregiver_dyad, surrogate_child_dyad, pair_type, channel_pair, edge_type, ff_dtf, group, event.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.path.abspath('..'))

from src.export import load_eeg_signals, load_xarray_from_netcdf, get_export_metadata
from src.mtmvar import mvar_criterion, full_freq_dtf, multivariate_spectra

In [ ]:
# Input folder with exported EEG NetCDF files
export_folder = Path("/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported")

# Which events to include
TARGET_EVENTS = ['Peppa', 'Brave', 'Incredibles']

# Same subset is used for child and caregiver
CHANNEL_SUBSET = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']
# CHANNEL_SUBSET = None  # use all channels common to child and caregiver

# Optional signal pre-filtering (before margin trimming inside load_eeg_signals)
SIGNAL_LOW_CUTOFF_HZ = None
SIGNAL_HIGH_CUTOFF_HZ = None

# MVAR settings
MAX_MODEL_ORDER = 25
OPTIMAL_MODEL_ORDER = 21  # set None to auto-search with CRIT_TYPE
CRIT_TYPE = 'HQ'  # 'AIC', 'HQ', or 'SC'

# Frequency grid for ffDTF computation
FREQ_MIN = 4.0
FREQ_MAX = 20.0
FREQ_STEP = 0.5

# Frequency band used for ffDTF summation
SUM_FREQ_MIN = 7.0
SUM_FREQ_MAX = 13.0

# Keep only inter-brain (child <-> caregiver) directed edges
KEEP_ONLY_CROSS_PERSON = True

# Surrogate settings (caregiver from dyad A + random child from dyad B, B != A)
INCLUDE_SURROGATES = True
SURROGATE_RANDOM_SEED = 42
# --- Surrogate construction mode ---
# Default: use all possible caregiver-child combinations excluding true pairs.
SURROGATE_USE_ALL = False
SURROGATE_SUBSET_SIZE = 530  # used only when SURROGATE_USE_ALL=False

# Balance options for random subset mode
SURROGATE_BALANCE_CHILD_GROUP = True
SURROGATE_BALANCE_STRICT = True

# Optional smoke mode
SMOKE_TEST = False
SMOKE_MAX_PAIRS = 12

# Optional plotting
PLOT_CRIT = False
DEBUG_PLOT_FF_DTF = False
DEBUG_PLOT_FF_DTF_ONLY_CROSS = False
DEBUG_PLOT_FF_DTF_GRID = False
DEBUG_PLOT_FF_DTF_GRID_MAX_CH = 24

In [ ]:
FILE_RE = re.compile(r'^(W_\d+)_EEG_(ch|cg)_(.+)$')

def discover_role_files(root: Path, target_events):
    files = sorted(p for p in root.rglob('*.nc') if '_EEG_' in p.name)
    pairs = {}
    for p in files:
        m = FILE_RE.match(p.stem)
        if m is None:
            continue
        dyad_id, role, event = m.group(1), m.group(2), m.group(3)
        if event not in target_events:
            continue
        key = (dyad_id, event)
        pairs.setdefault(key, {})[role] = p

    complete = []
    for (dyad_id, event), roles in sorted(pairs.items()):
        if 'ch' in roles and 'cg' in roles:
            complete.append((dyad_id, event, roles['ch'], roles['cg']))
    return complete

def _select_common_channels(ch_names_child, ch_names_caregiver, requested_subset):
    if requested_subset is None:
        return [ch for ch in ch_names_child if ch in set(ch_names_caregiver)]
    return [ch for ch in requested_subset if ch in set(ch_names_child) and ch in set(ch_names_caregiver)]

def _debug_plot_ffdtf_grid(freqs, ff_dtf, spectra, node_names, dyad_id, event, pair_type):
    ff_abs = np.abs(ff_dtf)
    sp_abs = np.abs(spectra)
    n_nodes = ff_abs.shape[0]

    max_off = np.nanmax(ff_abs) if np.isfinite(np.nanmax(ff_abs)) else 1.0
    max_diag = np.nanmax(np.diagonal(sp_abs, axis1=0, axis2=1))
    if not np.isfinite(max_off) or max_off <= 0:
        max_off = 1.0
    if not np.isfinite(max_diag) or max_diag <= 0:
        max_diag = 1.0

    fig, axs = plt.subplots(
        n_nodes, n_nodes,
        figsize=(max(8, n_nodes * 0.8), max(8, n_nodes * 0.8)),
        gridspec_kw={'wspace': 0, 'hspace': 0},
    )

    for i in range(n_nodes):
        for j in range(n_nodes):
            ax = axs[i, j] if n_nodes > 1 else axs
            if i != j:
                y = np.real(ff_abs[i, j, :])
                ax.plot(freqs, y, lw=0.5)
                ax.fill_between(freqs, y, 0, color='skyblue', alpha=0.4)
                ax.set_ylim([0, max_off])
            else:
                y = np.real(sp_abs[i, i, :])
                ax.plot(freqs, y, lw=0.5, color=[0.7, 0.7, 0.7])
                ax.fill_between(freqs, y, 0, color=[0.7, 0.7, 0.7], alpha=0.4)
                ax.set_ylim([0, max_diag])

            ax.tick_params(
                labelleft=(j == 0), labelbottom=(i == n_nodes - 1),
                left=(j == 0), bottom=(i == n_nodes - 1),
                labelsize=4,
            )
            if i == n_nodes - 1:
                ax.set_xlabel(node_names[j], fontsize=4)
            if j == 0:
                ax.set_ylabel(node_names[i], fontsize=4)

    fig.suptitle(
        f'ffDTF(freq) + spectra(diag) | {dyad_id} | {event} | {pair_type}',
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

def _debug_plot_ffdtf_sum_matrix(ff_dtf_sum, node_names, use_channels, dyad_id, event, pair_type):
    plot_mat = ff_dtf_sum.copy()

    # Always hide the diagonal (self-connections) in the summed matrix debug plot.
    np.fill_diagonal(plot_mat, np.nan)

    if DEBUG_PLOT_FF_DTF_ONLY_CROSS:
        n_ch = len(use_channels)
        mask = np.zeros_like(plot_mat, dtype=bool)
        mask[:n_ch, n_ch:] = True
        mask[n_ch:, :n_ch] = True
        plot_mat = np.where(mask, plot_mat, np.nan)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(plot_mat, cmap='viridis', aspect='auto')
    ax.set_title(f'ffDTF sum ({SUM_FREQ_MIN:g}-{SUM_FREQ_MAX:g} Hz) | {dyad_id} | {event} | {pair_type}')
    ticks = np.arange(len(node_names))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(node_names, rotation=90, fontsize=6)
    ax.set_yticklabels(node_names, fontsize=6)
    ax.set_xlabel('Source channel')
    ax.set_ylabel('Target channel')
    fig.colorbar(im, ax=ax, label='Summed ffDTF')
    plt.tight_layout()
    plt.show()

def compute_pair_ffdtf_rows(
    path_child: Path,
    path_caregiver: Path,
    dyad_id: str,
    event: str,
    pair_type: str,
    surrogate_pair_id=None,
    surrogate_caregiver_dyad=None,
    surrogate_child_dyad=None,
):
    sig_ch, names_ch, fs_ch, *_ = load_eeg_signals(
        str(path_child),
        channel_subset=CHANNEL_SUBSET,
        low_cutoff_hz=SIGNAL_LOW_CUTOFF_HZ,
        high_cutoff_hz=SIGNAL_HIGH_CUTOFF_HZ,
    )
    sig_cg, names_cg, fs_cg, *_ = load_eeg_signals(
        str(path_caregiver),
        channel_subset=CHANNEL_SUBSET,
        low_cutoff_hz=SIGNAL_LOW_CUTOFF_HZ,
        high_cutoff_hz=SIGNAL_HIGH_CUTOFF_HZ,
    )

    if not np.isclose(fs_ch, fs_cg):
        raise ValueError(f'Sampling mismatch for {dyad_id}/{event}: child={fs_ch}, caregiver={fs_cg}')

    use_channels = _select_common_channels(names_ch, names_cg, CHANNEL_SUBSET)
    if len(use_channels) == 0:
        raise ValueError(f'No common channels for {dyad_id}/{event}')

    idx_ch = [names_ch.index(ch) for ch in use_channels]
    idx_cg = [names_cg.index(ch) for ch in use_channels]

    sig_ch = sig_ch[idx_ch, :]
    sig_cg = sig_cg[idx_cg, :]

    n_samp = min(sig_ch.shape[1], sig_cg.shape[1])
    sig_ch = sig_ch[:, :n_samp]
    sig_cg = sig_cg[:, :n_samp]

    signals_joint = np.vstack([sig_ch, sig_cg])
    node_names = [f'ch:{ch}' for ch in use_channels] + [f'cg:{ch}' for ch in use_channels]

    freqs = np.arange(FREQ_MIN, FREQ_MAX + FREQ_STEP * 0.5, FREQ_STEP)
    band_mask = (freqs >= SUM_FREQ_MIN) & (freqs <= SUM_FREQ_MAX)
    if not np.any(band_mask):
        raise ValueError('Selected summation band does not overlap the ffDTF frequency grid')

    if OPTIMAL_MODEL_ORDER is None:
        _, _, p_opt = mvar_criterion(signals_joint, MAX_MODEL_ORDER, CRIT_TYPE, plot=PLOT_CRIT)
    else:
        p_opt = int(OPTIMAL_MODEL_ORDER)

    ff_dtf = full_freq_dtf(
        signals_joint,
        freqs=freqs,
        fs=float(fs_ch),
        max_model_order=MAX_MODEL_ORDER,
        optimal_model_order=p_opt,
        crit_type=CRIT_TYPE,
    )

    # Optional ESCan_drfat-style debug plot before ffDTF summation.
    if DEBUG_PLOT_FF_DTF_GRID and len(node_names) <= DEBUG_PLOT_FF_DTF_GRID_MAX_CH:
        spectra = multivariate_spectra(
            signals_joint,
            freqs=freqs,
            fs=float(fs_ch),
            max_model_order=MAX_MODEL_ORDER,
            optimal_model_order=p_opt,
            crit_type=CRIT_TYPE,
        )
        _debug_plot_ffdtf_grid(freqs, ff_dtf, spectra, node_names, dyad_id, event, pair_type)

    ff_dtf_sum = ff_dtf[:, :, band_mask].sum(axis=2)

    # Optional debugging plot for the current dyad/event (summed matrix).
    if DEBUG_PLOT_FF_DTF:
        _debug_plot_ffdtf_sum_matrix(ff_dtf_sum, node_names, use_channels, dyad_id, event, pair_type)

    # Group is always read from the child file metadata (real and surrogate).
    da_meta = load_xarray_from_netcdf(str(path_child))
    meta = get_export_metadata(da_meta)
    group = meta.get('child_info', {}).get('group', np.nan)

    if surrogate_pair_id is None:
        surrogate_pair_id = dyad_id
    if surrogate_caregiver_dyad is None:
        surrogate_caregiver_dyad = dyad_id
    if surrogate_child_dyad is None:
        surrogate_child_dyad = dyad_id

    rows = []
    n_nodes = len(node_names)
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i == j:
                continue
            src = node_names[j]
            dst = node_names[i]
            edge_type = 'cross-brain' if src.split(':')[0] != dst.split(':')[0] else 'intra-brain'
            if KEEP_ONLY_CROSS_PERSON and edge_type == 'intra-brain':
                continue
            rows.append({
                'dyadID': dyad_id,
                'surrogate_pair_id': surrogate_pair_id,
                'surrogate_caregiver_dyad': surrogate_caregiver_dyad,
                'surrogate_child_dyad': surrogate_child_dyad,
                'pair_type': pair_type,
                'channel_pair': f'{src}->{dst}',
                'edge_type': edge_type,
                'ff_dtf': float(ff_dtf_sum[i, j]),
                'group': group,
                'event': event,
            })

    debug_info = {
        'dyadID': dyad_id,
        'event': event,
        'pair_type': pair_type,
        'n_channels_per_person': len(use_channels),
        'p_opt': p_opt,
        'n_rows': len(rows),
    }
    return rows, debug_info

In [ ]:
# Scan data and build the list of dyads with complete EEG sets (all TARGET_EVENTS)
pairs_all = discover_role_files(export_folder, TARGET_EVENTS)

if not pairs_all:
    raise FileNotFoundError(f'No complete child/caregiver EEG pairs found in: {export_folder}')

events_required = set(TARGET_EVENTS)
events_by_dyad = {}
for dyad_id, event, _, _ in pairs_all:
    events_by_dyad.setdefault(dyad_id, set()).add(event)

good_dyads = sorted([
    dyad_id for dyad_id, evs in events_by_dyad.items()
    if events_required.issubset(evs)
])

# Optional manual exclusion after visual/quality inspection
MANUAL_EXCLUDE_DYADS = ['W_044', 'W_057']  # e.g. ['W_013', 'W_027']
if MANUAL_EXCLUDE_DYADS:
    good_dyads = [d for d in good_dyads if d not in set(MANUAL_EXCLUDE_DYADS)]

print('Dyads with complete EEG event set:')
for d in good_dyads:
    print(f'  - {d}')

print(f'\nGood dyads count: {len(good_dyads)}')
print(f'Required events: {sorted(events_required)}')

In [ ]:
# Build analysis set strictly from previously detected complete dyads
if not good_dyads:
    raise ValueError('No good dyads available after completeness check/manual exclusion.')

selected_dyads = list(good_dyads)
if SMOKE_TEST:
    selected_dyads = selected_dyads[:SMOKE_MAX_PAIRS]

pairs = [p for p in pairs_all if p[0] in set(selected_dyads)]
if not pairs:
    raise ValueError('No EEG pairs left for analysis after dyad filtering.')

print(f'Real dyads selected: {len(selected_dyads)}')
print('Selected dyads:')
for d in selected_dyads:
    print(f'  - {d}')

print(f'Real dyad-event pairs to process: {len(pairs)}')
for dyad_id, event, _, _ in pairs:
    print(f'  - {dyad_id} | {event}')

rng = np.random.default_rng(SURROGATE_RANDOM_SEED)

# Build quick lookup for available real files (restricted to selected complete dyads)
real_lookup = {(dyad_id, event): (path_ch, path_cg) for dyad_id, event, path_ch, path_cg in pairs}
target_events = list(TARGET_EVENTS)

# Selected dyads are already complete by construction
eligible_dyads = list(selected_dyads)
print(f'Eligible dyads for surrogate construction: {len(eligible_dyads)}')

# Build child-group lookup by dyad (group label is child-derived in downstream rows)
child_group_by_dyad = {}
for dyad_id in eligible_dyads:
    path_ch, _ = real_lookup[(dyad_id, target_events[0])]
    da_meta = load_xarray_from_netcdf(str(path_ch))
    meta = get_export_metadata(da_meta)
    group = str(meta.get('child_info', {}).get('group', '')).strip().upper()
    child_group_by_dyad[dyad_id] = group

eligible_group_counts = pd.Series(child_group_by_dyad).value_counts(dropna=False)
print('Eligible child-group counts:')
print(eligible_group_counts)



# Build all ordered combinations caregiver x child, excluding dyad_cg == dyad_ch.
all_surrogate_dyads = [
    (dyad_cg, dyad_ch)
    for dyad_cg in eligible_dyads
    for dyad_ch in eligible_dyads
    if dyad_cg != dyad_ch
]

surrogate_pairs = []
selected_surrogate_dyads = []
if INCLUDE_SURROGATES:
    if len(eligible_dyads) < 2:
        print('Surrogate generation skipped: need at least 2 eligible dyads with all target events.')
    elif not all_surrogate_dyads:
        print('Surrogate generation skipped: no valid caregiver-child combinations found.')
    else:
        if SURROGATE_USE_ALL:
            selected_surrogate_dyads = list(all_surrogate_dyads)
            print('Surrogate mode: ALL combinations')
        else:
            if SURROGATE_SUBSET_SIZE is None:
                raise ValueError('Set SURROGATE_SUBSET_SIZE when SURROGATE_USE_ALL=False.')

            n_available = len(all_surrogate_dyads)
            n_pick = int(SURROGATE_SUBSET_SIZE)
            if n_pick <= 0:
                raise ValueError('SURROGATE_SUBSET_SIZE must be > 0.')
            if n_pick > n_available:
                print(f'Requested subset {n_pick} exceeds available {n_available}; using all available.')
                n_pick = n_available

            if SURROGATE_BALANCE_CHILD_GROUP:
                td_candidates = [p for p in all_surrogate_dyads if child_group_by_dyad.get(p[1]) == 'TD']
                asd_candidates = [p for p in all_surrogate_dyads if child_group_by_dyad.get(p[1]) == 'ASD']

                if SURROGATE_BALANCE_STRICT and (n_pick % 2 != 0):
                    raise ValueError(
                        'SURROGATE_SUBSET_SIZE must be even for strict TD/ASD balancing.'
                    )

                max_balanced = 2 * min(len(td_candidates), len(asd_candidates))
                if max_balanced == 0:
                    raise ValueError('Cannot balance TD/ASD: one group has zero candidates.')

                if SURROGATE_BALANCE_STRICT and n_pick > max_balanced:
                    raise ValueError(
                        f'Requested subset {n_pick} exceeds max strictly balanced size {max_balanced}. '
                        'Reduce SURROGATE_SUBSET_SIZE or disable strict balancing.'
                    )
                if n_pick > max_balanced:
                    print(f'Requested subset {n_pick} exceeds balanced capacity {max_balanced}; using {max_balanced}.')
                    n_pick = max_balanced

                n_each = n_pick // 2
                idx_td = rng.choice(len(td_candidates), size=n_each, replace=False)
                idx_asd = rng.choice(len(asd_candidates), size=n_each, replace=False)

                selected_surrogate_dyads = [td_candidates[i] for i in idx_td] + [asd_candidates[i] for i in idx_asd]

                if (n_pick % 2 != 0) and (not SURROGATE_BALANCE_STRICT):
                    leftovers = [
                        p for p in all_surrogate_dyads
                        if p not in set(selected_surrogate_dyads)
                    ]
                    if leftovers:
                        selected_surrogate_dyads.append(leftovers[int(rng.integers(0, len(leftovers)))])

                selected_surrogate_dyads = list(selected_surrogate_dyads)
                rng.shuffle(selected_surrogate_dyads)
                print(f'Surrogate mode: BALANCED RANDOM SUBSET ({len(selected_surrogate_dyads)} of {n_available})')
            else:
                idx = rng.choice(n_available, size=n_pick, replace=False)
                selected_surrogate_dyads = [all_surrogate_dyads[i] for i in idx]
                print(f'Surrogate mode: RANDOM SUBSET ({n_pick} of {n_available})')

        # Expand selected surrogate dyads to all events
        for dyad_cg, dyad_ch in selected_surrogate_dyads:
            surrogate_id = f'S_{dyad_cg}_cg__{dyad_ch}_ch'
            for event in target_events:
                path_ch, _ = real_lookup[(dyad_ch, event)]
                _, path_cg = real_lookup[(dyad_cg, event)]
                surrogate_pairs.append((surrogate_id, event, path_ch, path_cg, dyad_cg, dyad_ch))

selected_child_groups = [child_group_by_dyad.get(dyad_ch, np.nan) for _, dyad_ch in selected_surrogate_dyads]
selected_group_counts = pd.Series(selected_child_groups).value_counts(dropna=False) if selected_child_groups else pd.Series(dtype=int)

print(f'All possible surrogate caregiver-child dyads: {len(all_surrogate_dyads)}')
print(f'Selected surrogate caregiver-child dyads: {len(selected_surrogate_dyads)}')
if not selected_group_counts.empty:
    print('Selected surrogate child-group counts:')
    print(selected_group_counts)
print(f'Surrogate dyad-event pairs to process: {len(surrogate_pairs)}')

if surrogate_pairs:
    preview_n = min(40, len(surrogate_pairs))
    print(f'Preview first {preview_n} surrogate dyad-event pairs:')
    for surrogate_id, event, _, _, dyad_cg, dyad_ch in surrogate_pairs[:preview_n]:
        print(f'  - {surrogate_id} | {event} (cg={dyad_cg}, ch={dyad_ch})')
    if len(surrogate_pairs) > preview_n:
        print(f'  ... ({len(surrogate_pairs) - preview_n} more)')

all_rows = []
debug_rows = []
failed = []

# Real dyads
for dyad_id, event, path_ch, path_cg in pairs:
    print(f'Processing REAL {dyad_id} | {event}')
    try:
        rows, dbg = compute_pair_ffdtf_rows(
            path_child=path_ch,
            path_caregiver=path_cg,
            dyad_id=dyad_id,
            event=event,
            pair_type='real',
            surrogate_pair_id=dyad_id,
            surrogate_caregiver_dyad=dyad_id,
            surrogate_child_dyad=dyad_id,
        )
        all_rows.extend(rows)
        debug_rows.append(dbg)
        print(f"  OK: rows={dbg['n_rows']}, channels/person={dbg['n_channels_per_person']}, p={dbg['p_opt']}")
    except Exception as exc:
        failed.append({'dyadID': dyad_id, 'event': event, 'pair_type': 'real', 'error': str(exc)})
        print(f'  FAILED: {exc}')

# Surrogate dyads
for surrogate_id, event, path_ch, path_cg, dyad_cg, dyad_ch in surrogate_pairs:
    print(f'Processing SURROGATE {surrogate_id} | {event}')
    try:
        rows, dbg = compute_pair_ffdtf_rows(
            path_child=path_ch,
            path_caregiver=path_cg,
            dyad_id=surrogate_id,
            event=event,
            pair_type='surrogate',
            surrogate_pair_id=surrogate_id,
            surrogate_caregiver_dyad=dyad_cg,
            surrogate_child_dyad=dyad_ch,
        )
        all_rows.extend(rows)
        debug_rows.append(dbg)
        print(f"  OK: rows={dbg['n_rows']}, channels/person={dbg['n_channels_per_person']}, p={dbg['p_opt']}")
    except Exception as exc:
        failed.append({'dyadID': surrogate_id, 'event': event, 'pair_type': 'surrogate', 'error': str(exc)})
        print(f'  FAILED: {exc}')

ffdtf_df = pd.DataFrame(
    all_rows,
    columns=[
        'dyadID',
        'surrogate_pair_id',
        'surrogate_caregiver_dyad',
        'surrogate_child_dyad',
        'pair_type',
        'channel_pair',
        'edge_type',
        'ff_dtf',
        'group',
        'event',
    ],
)
debug_df = pd.DataFrame(debug_rows)
failed_df = pd.DataFrame(failed)

print('\nSummary')
print(f'  Total rows: {len(ffdtf_df)}')
print(f'  Successful jobs: {len(debug_df)}')
print(f'  Failed jobs: {len(failed_df)}')
if not ffdtf_df.empty:
    print('  Group counts:')
    print(ffdtf_df['group'].value_counts(dropna=False))

display(ffdtf_df.head(20))
if not failed_df.empty:
    display(failed_df)

In [ ]:
output_dir = Path('../../EEG_DTF_tables')
output_dir.mkdir(parents=True, exist_ok=True)

band_tag = f'{SUM_FREQ_MIN:g}-{SUM_FREQ_MAX:g}Hz'.replace('.', 'p')
suffix = 'cross_only' if KEEP_ONLY_CROSS_PERSON else 'all_edges'
output_csv = output_dir / f'ffdtf_batch_{band_tag}_{suffix}.csv'

ffdtf_df.to_csv(output_csv, index=False)
print(f'Saved: {output_csv}')
print(f'Rows: {len(ffdtf_df)}')

In [ ]:
# Repeated-measures completeness check by surrogate_pair_id × event
if ffdtf_df.empty:
    print('ffdtf_df is empty.')
else:
    counts_by_pair_event = (
        ffdtf_df.groupby(['surrogate_pair_id', 'event'])
        .size()
        .unstack(fill_value=0)
    )
    
    event_nunique = ffdtf_df.groupby('surrogate_pair_id')['event'].nunique()
    required_n_events = len(TARGET_EVENTS)
    complete_pairs = event_nunique[event_nunique == required_n_events].index
    incomplete_pairs = event_nunique[event_nunique < required_n_events].index
    
    print('Repeated-measures summary')
    print(f'  Unique surrogate_pair_id: {event_nunique.shape[0]}')
    print(f'  Pairs with all {required_n_events} events: {len(complete_pairs)}')
    print(f'  Pairs with missing events: {len(incomplete_pairs)}')
    
    display(counts_by_pair_event.head(30))
    
    if len(incomplete_pairs) > 0:
        print('\nPairs with missing events:')
        display(counts_by_pair_event.loc[incomplete_pairs])

## Statistics preprocessing

In [ ]:
# Preprocess dataframe for statistical analysis
TRANSFORM_METHOD = 'arcsin_sqrt'  # 'none', 'arctanh', 'arcsin_sqrt'

if ffdtf_df.empty:
    print('ffdtf_df is empty. Run the batch computation first.')
    ffdtf_df_pre = ffdtf_df.copy()
else:
    df = ffdtf_df.copy()

    n_before = len(df)

    # Remove edges containing Fp1/Fp2 on either side (child or caregiver).
    fp_mask = df['channel_pair'].str.contains(r':Fp1|:Fp2', regex=True, na=False)
    df = df.loc[~fp_mask].copy()

    # Keep only target groups for downstream statistical analyses.
    allowed_groups = {'TD', 'ASD'}
    group_clean = df['group'].astype(str).str.strip().str.upper()
    df = df.loc[group_clean.isin(allowed_groups)].copy()

    # Optional transformation of DTF values for downstream analysis.
    # arctanh: z = arctanh(DTF)
    # arcsin_sqrt: z = arcsin(sqrt(DTF))
    method = str(TRANSFORM_METHOD).strip().lower()
    df['ff_dtf_raw'] = df['ff_dtf']

    if method == 'none':
        pass
    elif method == 'arctanh':
        # Keep within open interval (-1, 1) for stable arctanh.
        x = np.clip(df['ff_dtf_raw'].to_numpy(dtype=float), -0.999999, 0.999999)
        df['ff_dtf'] = np.arctanh(x)
    elif method == 'arcsin_sqrt':
        # Requires [0, 1]; clip to numeric bounds before sqrt/arcsin.
        x = np.clip(df['ff_dtf_raw'].to_numpy(dtype=float), 0.0, 1.0)
        df['ff_dtf'] = np.arcsin(np.sqrt(x))
    else:
        raise ValueError("Invalid TRANSFORM_METHOD. Use 'none', 'arctanh', or 'arcsin_sqrt'.")

    ffdtf_df_pre = df

    print('Preprocessing summary:')
    print(f'  Rows before: {n_before}')
    print(f'  Rows after:  {len(ffdtf_df_pre)}')
    print(f'  Removed rows: {n_before - len(ffdtf_df_pre)}')
    print('  Groups kept:', sorted(allowed_groups))
    print(f'  Transform method: {method}')
    if not ffdtf_df_pre.empty:
        print('  Group counts after filtering:')
        print(ffdtf_df_pre['group'].value_counts(dropna=False))
        print('  Dyad type counts after filtering:')
        print(ffdtf_df_pre['pair_type'].value_counts(dropna=False))

In [ ]:
# Split-violin grid: ff_dtf distributions per edge (real vs surrogate), separately per movie
from matplotlib.patches import Rectangle

def _split_channel_pair(cp):
    src, dst = cp.split('->', 1)
    return src, dst

def _add_half_violin(ax, values, side='left', color='#4C72B0', alpha=0.65):
    if values is None or len(values) < 2:
        return
    vp = ax.violinplot(
        [values], positions=[0], widths=0.9,
        showmeans=False, showmedians=False, showextrema=False
    )
    body = vp['bodies'][0]
    body.set_facecolor(color)
    body.set_edgecolor('black')
    body.set_linewidth(0.4)
    body.set_alpha(alpha)

    # Clip one half of the violin around x=0.
    if side == 'left':
        clip = Rectangle((-1.0, -1e9), 1.0, 2e9, transform=ax.transData)
    else:
        clip = Rectangle((0.0, -1e9), 1.0, 2e9, transform=ax.transData)
    body.set_clip_path(clip)

if 'ffdtf_df_pre' not in globals():
    print('ffdtf_df_pre not found. Run preprocessing cell first.')
else:
    vis_df_all = ffdtf_df_pre[
        (ffdtf_df_pre['edge_type'] == 'cross-brain') &
        (ffdtf_df_pre['pair_type'].isin(['real', 'surrogate']))
    ].copy()

    if vis_df_all.empty:
        print('No rows available for plotting after filters.')
    else:
        event_order = [ev for ev in TARGET_EVENTS if ev in set(vis_df_all['event'].unique())]
        if len(event_order) == 0:
            print('No matching events found in filtered data.')
        else:
            for ev in event_order:
                vis_df = vis_df_all.loc[vis_df_all['event'] == ev].copy()
                if vis_df.empty:
                    print(f'Event {ev}: no rows after filtering, skipping.')
                    continue

                # Parse source/target channel labels from channel_pair
                vis_df[['src', 'dst']] = vis_df['channel_pair'].apply(
                    lambda s: pd.Series(_split_channel_pair(s))
                )

                node_order = sorted(set(vis_df['src']).union(set(vis_df['dst'])))
                n = len(node_order)
                idx = {name: i for i, name in enumerate(node_order)}

                grouped = vis_df.groupby(['src', 'dst', 'pair_type'])['ff_dtf']
                real_map = {}
                sur_map = {}
                for (src, dst, pair_type), vals in grouped:
                    key = (idx[src], idx[dst])
                    if pair_type == 'real':
                        real_map[key] = vals.to_numpy()
                    else:
                        sur_map[key] = vals.to_numpy()

                # Robust y-axis limit for comparability across cells (within movie)
                y_vals = vis_df['ff_dtf'].to_numpy()
                y_max = np.nanpercentile(y_vals, 99.5) if len(y_vals) else 1.0
                if not np.isfinite(y_max) or y_max <= 0:
                    y_max = np.nanmax(y_vals) if len(y_vals) else 1.0
                if not np.isfinite(y_max) or y_max <= 0:
                    y_max = 1.0

                fig, axs = plt.subplots(
                    n, n,
                    figsize=(max(10, n * 0.65), max(10, n * 0.65)),
                    gridspec_kw={'wspace': 0.0, 'hspace': 0.0},
                    squeeze=False,
                )

                for i in range(n):
                    for j in range(n):
                        ax = axs[i, j]
                        k = (i, j)

                        _add_half_violin(ax, real_map.get(k), side='left', color='#2C7FB8', alpha=0.70)
                        _add_half_violin(ax, sur_map.get(k), side='right', color='#D95F0E', alpha=0.70)

                        ax.set_xlim(-0.6, 0.6)
                        ax.set_ylim(0, y_max)
                        ax.set_xticks([])

                        # Show y ticks only in first column
                        if j == 0:
                            ax.tick_params(axis='y', labelsize=4, length=1)
                        else:
                            ax.set_yticks([])

                        # Label outer axes only
                        if i == n - 1:
                            ax.set_xlabel(node_order[j], fontsize=4)
                        if j == 0:
                            ax.set_ylabel(node_order[i], fontsize=4)

                fig.suptitle(
                    f'ff_dtf distributions per edge (cross-brain), event={ev}\nleft half = real, right half = surrogate',
                    fontsize=11
                )

                legend_handles = [
                    Rectangle((0, 0), 1, 1, facecolor='#2C7FB8', edgecolor='black', label='real'),
                    Rectangle((0, 0), 1, 1, facecolor='#D95F0E', edgecolor='black', label='surrogate'),
                ]
                fig.legend(handles=legend_handles, loc='upper right', fontsize=8)

                plt.tight_layout()
                plt.show()

## Statistical Test 1: Real vs Surrogate (Cross-brain Edges)

This test evaluates whether any cross-brain edge differs significantly between real dyads and surrogate dyads.

Method summary:
- Keep only rows with edge_type = cross-brain and pair_type in {real, surrogate}.
- Aggregate ff_dtf within each pair and edge across events (mean over movies), to avoid pseudoreplication from repeated measures.
- For each channel_pair, run a two-sided permutation test on the difference of means (real minus surrogate).
- Apply Benjamini-Hochberg FDR correction across all tested edges.

Interpretation:
- The key output is Any significant edge?: True/False.
- If True, at least one cross-brain edge remains significant after FDR correction at alpha = 0.05.

In [ ]:
# Test 1 utilities: edge-wise real vs surrogate permutation test
import numpy as np
import pandas as pd

# --- Default config for statistical testing ---
ALPHA = 0.05
N_PERM = 5000
MIN_PAIRS_PER_GROUP = 5
RANDOM_SEED_TEST = 123

def _bh_fdr(p_values):
    """Benjamini-Hochberg FDR correction. Returns q-values in original order."""
    p = np.asarray(p_values, dtype=float)
    n = p.size
    order = np.argsort(p)
    p_sorted = p[order]
    q_sorted = np.empty(n, dtype=float)

    # Raw BH adjusted values
    for i, pv in enumerate(p_sorted, start=1):
        q_sorted[i - 1] = pv * n / i

    # Enforce monotonicity from the end
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0.0, 1.0)

    q = np.empty(n, dtype=float)
    q[order] = q_sorted
    return q

def _perm_pvalue_two_sided(x, y, n_perm=5000, rng=None):
    """Permutation p-value for difference in means (two-sided)."""
    if rng is None:
        rng = np.random.default_rng(0)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    obs = np.mean(x) - np.mean(y)
    pooled = np.concatenate([x, y])
    n_x = x.size

    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        d = np.mean(pooled[:n_x]) - np.mean(pooled[n_x:])
        if abs(d) >= abs(obs):
            count += 1

    # +1 correction to avoid 0 p-values
    p = (count + 1) / (n_perm + 1)
    return p, obs

def run_edgewise_real_vs_surrogate_permutation(
    df_input,
    alpha=ALPHA,
    n_perm=N_PERM,
    min_pairs_per_group=MIN_PAIRS_PER_GROUP,
    random_seed=RANDOM_SEED_TEST,
    aggregate_over_events=True,
    filter_cross_brain=True,
    pair_types=('real', 'surrogate'),
):
    """Run edge-wise permutation tests (real vs surrogate) on a provided dataframe subset.

    Parameters
    ----------
    df_input : pd.DataFrame
        Input dataframe (can be full data or any subset).
    aggregate_over_events : bool
        If True, aggregates ff_dtf within pair and edge over events before testing.
    filter_cross_brain : bool
        If True, keeps only edge_type == 'cross-brain'.
    pair_types : tuple[str, str]
        Names of the two groups to compare, default ('real', 'surrogate').
    """
    if df_input is None or len(df_input) == 0:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'Input dataframe is empty.'
        }

    g1, g2 = pair_types
    test_df = df_input.copy()

    if filter_cross_brain:
        test_df = test_df.loc[test_df['edge_type'] == 'cross-brain'].copy()

    test_df = test_df.loc[test_df['pair_type'].isin([g1, g2])].copy()
    if test_df.empty:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'No rows after filtering edge_type/pair_type.'
        }

    if aggregate_over_events:
        pair_level = (
            test_df.groupby(['surrogate_pair_id', 'pair_type', 'channel_pair'], as_index=False)['ff_dtf']
            .mean()
            .rename(columns={'ff_dtf': 'ff_dtf_stat'})
        )
    else:
        pair_level = test_df.rename(columns={'ff_dtf': 'ff_dtf_stat'})

    rng = np.random.default_rng(random_seed)
    results = []

    for edge, g in pair_level.groupby('channel_pair'):
        x = g.loc[g['pair_type'] == g1, 'ff_dtf_stat'].to_numpy()
        y = g.loc[g['pair_type'] == g2, 'ff_dtf_stat'].to_numpy()

        if x.size < min_pairs_per_group or y.size < min_pairs_per_group:
            continue

        p_val, diff_mean = _perm_pvalue_two_sided(x, y, n_perm=n_perm, rng=rng)
        results.append({
            'channel_pair': edge,
            f'n_{g1}': int(x.size),
            f'n_{g2}': int(y.size),
            f'mean_{g1}': float(np.mean(x)),
            f'mean_{g2}': float(np.mean(y)),
            f'diff_{g1}_minus_{g2}': float(diff_mean),
            'p_perm': float(p_val),
        })

    if len(results) == 0:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'No edges met min_pairs_per_group criterion.'
        }

    edge_test_df = pd.DataFrame(results).sort_values('p_perm').reset_index(drop=True)
    edge_test_df['q_fdr_bh'] = _bh_fdr(edge_test_df['p_perm'].to_numpy())
    edge_test_df['significant_fdr'] = edge_test_df['q_fdr_bh'] < alpha

    n_sig = int(edge_test_df['significant_fdr'].sum())
    summary = {
        'tested_edges': int(len(edge_test_df)),
        'n_significant_fdr': n_sig,
        'any_significant': bool(n_sig > 0),
        'message': 'OK'
    }
    return edge_test_df, summary

# Example run on the full preprocessed dataset
if ffdtf_df_pre.empty:
    print('ffdtf_df_pre is empty. Run batch + preprocessing first.')
else:
    edge_test_df, edge_test_summary = run_edgewise_real_vs_surrogate_permutation(
        ffdtf_df_pre,
        alpha=ALPHA,
        n_perm=N_PERM,
        min_pairs_per_group=MIN_PAIRS_PER_GROUP,
        random_seed=RANDOM_SEED_TEST,
        aggregate_over_events=True,
        filter_cross_brain=True,
        pair_types=('real', 'surrogate'),
    )

    print('Edge-wise cross-brain test (full data): real vs surrogate')
    print(f"  Tested edges: {edge_test_summary['tested_edges']}")
    print(f"  Significant after BH-FDR (alpha={ALPHA}): {edge_test_summary['n_significant_fdr']}")
    print(f"  Any significant edge?: {edge_test_summary['any_significant']}")
    if edge_test_summary['message'] != 'OK':
        print(f"  Note: {edge_test_summary['message']}")

    if not edge_test_df.empty:
        display(edge_test_df.head(30))
        if edge_test_summary['any_significant']:
            print('\nSignificant edges (FDR):')
            display(edge_test_df.loc[edge_test_df['significant_fdr']].sort_values('q_fdr_bh'))

In [ ]:
# Example: external loop over events using the same permutation-test function
if ffdtf_df_pre.empty:
    print('ffdtf_df_pre is empty. Run batch + preprocessing first.')
else:
    event_test_results = {}
    for ev in TARGET_EVENTS:
        df_ev = ffdtf_df_pre.loc[ffdtf_df_pre['event'] == ev].copy()
        edge_df_ev, summary_ev = run_edgewise_real_vs_surrogate_permutation(
            df_ev,
            alpha=ALPHA,
            n_perm=N_PERM,
            min_pairs_per_group=MIN_PAIRS_PER_GROUP,
            random_seed=RANDOM_SEED_TEST,
            aggregate_over_events=False,  # per-event test
            filter_cross_brain=True,
            pair_types=('real', 'surrogate'),
        )
        event_test_results[ev] = {'results': edge_df_ev, 'summary': summary_ev}

        print('\n' + '=' * 72)
        print(f'Event: {ev}')
        print(f"  Tested edges: {summary_ev['tested_edges']}")
        print(f"  Significant after BH-FDR (alpha={ALPHA}): {summary_ev['n_significant_fdr']}")
        print(f"  Any significant edge?: {summary_ev['any_significant']}")
        if summary_ev['message'] != 'OK':
            print(f"  Note: {summary_ev['message']}")
        if not edge_df_ev.empty:
            display(edge_df_ev.head(20))

## Statistical Test 2: Real vs Surrogate (Cross-brain Edges, Distribution Shape)

This test evaluates whether cross-brain edges differ in **distribution shape** (not only mean) between real and surrogate dyads.

Method summary:
- Keep rows with edge_type = cross-brain and pair_type in {real, surrogate}.
- Optionally aggregate ff_dtf within pair and edge across events (mean over movies).
- For each channel_pair, compute a two-sample KS statistic and obtain permutation p-value by label shuffling.
- Apply Benjamini-Hochberg FDR correction across all tested edges.

Interpretation:
- A significant edge means the real and surrogate distributions differ in shape/location for that edge.
- Key output: number of FDR-significant edges and list of significant channel_pairs.

In [ ]:
# Test 2: edge-wise distribution-shape difference (KS + permutation)

# --- Default config for Test 2 ---
ALPHA_SHAPE = 0.05
N_PERM_SHAPE = 5000
MIN_PAIRS_PER_GROUP_SHAPE = 5
RANDOM_SEED_SHAPE = 321

def _ks_statistic_two_sample(x, y):
    """Two-sample KS statistic computed from empirical CDFs."""
    x = np.sort(np.asarray(x, dtype=float))
    y = np.sort(np.asarray(y, dtype=float))
    if x.size == 0 or y.size == 0:
        return np.nan

    grid = np.sort(np.unique(np.concatenate([x, y])))
    cdf_x = np.searchsorted(x, grid, side='right') / x.size
    cdf_y = np.searchsorted(y, grid, side='right') / y.size
    return float(np.max(np.abs(cdf_x - cdf_y)))

def _perm_pvalue_ks(x, y, n_perm=5000, rng=None):
    """Permutation p-value for two-sample KS statistic."""
    if rng is None:
        rng = np.random.default_rng(0)

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_x = x.size
    pooled = np.concatenate([x, y])

    obs = _ks_statistic_two_sample(x, y)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        x_perm = pooled[:n_x]
        y_perm = pooled[n_x:]
        stat_perm = _ks_statistic_two_sample(x_perm, y_perm)
        if stat_perm >= obs:
            count += 1

    p = (count + 1) / (n_perm + 1)
    return p, obs

def run_edgewise_real_vs_surrogate_shape_test(
    df_input,
    alpha=ALPHA_SHAPE,
    n_perm=N_PERM_SHAPE,
    min_pairs_per_group=MIN_PAIRS_PER_GROUP_SHAPE,
    random_seed=RANDOM_SEED_SHAPE,
    aggregate_over_events=True,
    filter_cross_brain=True,
    pair_types=('real', 'surrogate'),
):
    """Run edge-wise permutation KS tests (shape difference) on provided dataframe subset."""
    if df_input is None or len(df_input) == 0:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'Input dataframe is empty.'
        }

    g1, g2 = pair_types
    test_df = df_input.copy()

    if filter_cross_brain:
        test_df = test_df.loc[test_df['edge_type'] == 'cross-brain'].copy()

    test_df = test_df.loc[test_df['pair_type'].isin([g1, g2])].copy()
    if test_df.empty:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'No rows after filtering edge_type/pair_type.'
        }

    if aggregate_over_events:
        pair_level = (
            test_df.groupby(['surrogate_pair_id', 'pair_type', 'channel_pair'], as_index=False)['ff_dtf']
            .mean()
            .rename(columns={'ff_dtf': 'ff_dtf_stat'})
        )
    else:
        pair_level = test_df.rename(columns={'ff_dtf': 'ff_dtf_stat'})

    rng = np.random.default_rng(random_seed)
    results = []

    for edge, g in pair_level.groupby('channel_pair'):
        x = g.loc[g['pair_type'] == g1, 'ff_dtf_stat'].to_numpy()
        y = g.loc[g['pair_type'] == g2, 'ff_dtf_stat'].to_numpy()

        if x.size < min_pairs_per_group or y.size < min_pairs_per_group:
            continue

        p_val, ks_stat = _perm_pvalue_ks(x, y, n_perm=n_perm, rng=rng)
        results.append({
            'channel_pair': edge,
            f'n_{g1}': int(x.size),
            f'n_{g2}': int(y.size),
            f'median_{g1}': float(np.median(x)),
            f'median_{g2}': float(np.median(y)),
            'ks_stat': float(ks_stat),
            'p_perm': float(p_val),
        })

    if len(results) == 0:
        return pd.DataFrame(), {
            'tested_edges': 0,
            'n_significant_fdr': 0,
            'any_significant': False,
            'message': 'No edges met min_pairs_per_group criterion.'
        }

    shape_test_df = pd.DataFrame(results).sort_values('p_perm').reset_index(drop=True)
    shape_test_df['q_fdr_bh'] = _bh_fdr(shape_test_df['p_perm'].to_numpy())
    shape_test_df['significant_fdr'] = shape_test_df['q_fdr_bh'] < alpha

    n_sig = int(shape_test_df['significant_fdr'].sum())
    summary = {
        'tested_edges': int(len(shape_test_df)),
        'n_significant_fdr': n_sig,
        'any_significant': bool(n_sig > 0),
        'message': 'OK'
    }
    return shape_test_df, summary

# Example run on the full preprocessed dataset
if ffdtf_df_pre.empty:
    print('ffdtf_df_pre is empty. Run batch + preprocessing first.')
else:
    shape_test_df, shape_test_summary = run_edgewise_real_vs_surrogate_shape_test(
        ffdtf_df_pre,
        alpha=ALPHA_SHAPE,
        n_perm=N_PERM_SHAPE,
        min_pairs_per_group=MIN_PAIRS_PER_GROUP_SHAPE,
        random_seed=RANDOM_SEED_SHAPE,
        aggregate_over_events=True,
        filter_cross_brain=True,
        pair_types=('real', 'surrogate'),
    )

    print('Test 2 (distribution shape, KS + permutation): real vs surrogate')
    print(f"  Tested edges: {shape_test_summary['tested_edges']}")
    print(f"  Significant after BH-FDR (alpha={ALPHA_SHAPE}): {shape_test_summary['n_significant_fdr']}")
    print(f"  Any significant edge?: {shape_test_summary['any_significant']}")
    if shape_test_summary['message'] != 'OK':
        print(f"  Note: {shape_test_summary['message']}")

    if not shape_test_df.empty:
        display(shape_test_df.head(30))
        if shape_test_summary['any_significant']:
            print('\nSignificant edges (FDR):')
            display(shape_test_df.loc[shape_test_df['significant_fdr']].sort_values('q_fdr_bh'))